# 03 · Análisis: rendimiento clutch ajustado por dificultad

**Pregunta principal:** ¿Existen jugadores de la NBA cuyo rendimiento en situaciones clutch sea estadísticamente superior al esperado, considerando la dificultad de sus lanzamientos, o las diferencias observadas pueden explicarse por la variación estadística?

Este notebook usa las mismas funciones que `python -m src.analysis.run_analysis`, para que el resultado sea reproducible fuera del notebook.

Requisito previo: haber ejecutado `src.ingestion.download_data` y `src.processing.build_dataset`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Identificación de tiros clutch

Definición (igual que en el notebook de Databricks):
- **Ventana final:** últimos 5 minutos del 4.º periodo o cualquier tiempo extra.
- **Marcador estimado:** último marcador conocido *antes* del instante del tiro (la fuente solo registra marcador después de encestes).
- **Clutch:** ventana final y diferencia estimada ≤ 5 puntos.
- Se excluyen instantes ambiguos (varios tiros en el mismo segundo) y tiros sin marcador previo.

In [2]:
from src.analysis.clutch import load_shot_detail, add_clutch_flags, analysis_base

df = add_clutch_flags(load_shot_detail())
base = analysis_base(df)

print(f"Tiros procesados:            {len(df):,}")
print(f"Tiros en ventana final:      {df['ventana_final'].sum():,}")
print(f"Instantes ambiguos excluidos:{df['instante_ambiguo'].sum():,}")
print(f"Base comparable:             {len(base):,}")
print(f"  Clutch:                    {base['clutch'].sum():,}")
print(f"  No clutch:                 {(~base['clutch']).sum():,}")

Tiros procesados:            703,781
Tiros en ventana final:      74,122
Instantes ambiguos excluidos:2,352
Base comparable:             692,926
  Clutch:                    23,483
  No clutch:                 669,443


## 2. Comparación bruta clutch vs. no clutch

In [3]:
from src.analysis.clutch import two_proportion_ci

g = base.groupby("grupo").agg(tiros=("is_made", "size"), encestes=("is_made", "sum"),
                              puntos=("puntos", "sum"))
g["efectividad"] = g["encestes"] / g["tiros"]
g["puntos_por_tiro"] = g["puntos"] / g["tiros"]
display(g)

diff, lo, hi = two_proportion_ci(g.loc["Clutch", "encestes"], g.loc["Clutch", "tiros"],
                                 g.loc["No clutch", "encestes"], g.loc["No clutch", "tiros"])
print(f"Diferencia bruta: {100*diff:.2f} pp  (IC 95%: {100*lo:.2f} a {100*hi:.2f})")

,tiros,encestes,puntos,efectividad,puntos_por_tiro
grupo,,,,,
Clutch,23483,10228,23530,0.4355,1.0020
No clutch,669443,315707,731288,0.4716,1.0924


Diferencia bruta: -3.60 pp  (IC 95%: -4.25 a -2.96)


## 3. ¿Cambia la selección de tiro? (pregunta secundaria)

Si en clutch se lanzan tiros más difíciles, la caída bruta no sería atribuible a la presión.

In [4]:
from scipy import stats
from src.analysis.run_analysis import range_comparison

t = range_comparison(base)
display(t[["rango_distancia", "grupo", "tiros", "proporcion_de_tiros", "efectividad", "puntos_por_tiro"]])

ks = stats.ks_2samp(base.loc[base["clutch"], "shot_distance"],
                    base.loc[~base["clutch"], "shot_distance"])
print(f"KS distancia: estadístico={ks.statistic:.4f}, p={ks.pvalue:.2e}")
print("Con muestras tan grandes, KS detecta diferencias mínimas: la mezcla de distancias es casi igual.")

,rango_distancia,grupo,tiros,proporcion_de_tiros,efectividad,puntos_por_tiro
0,0-4,Clutch,6891,0.2934,0.6253,1.2506
1,0-4,No clutch,203922,0.3046,0.6545,1.3091
6,5-14,Clutch,4956,0.2110,0.4203,0.8406
7,5-14,No clutch,142549,0.2129,0.4396,0.8791
2,15-24,Clutch,4377,0.1864,0.3655,0.9225
3,15-24,No clutch,122385,0.1828,0.3975,1.0315
4,25+,Clutch,7259,0.3091,0.3080,0.9241
5,25+,No clutch,200587,0.2996,0.3536,1.0608


KS distancia: estadístico=0.0232, p=5.00e-11
Con muestras tan grandes, KS detecta diferencias mínimas: la mezcla de distancias es casi igual.


## 4. Modelo de dificultad del tiro (regresión logística)

Se entrena **solo con tiros no clutch** para estimar la probabilidad de enceste de un tiro equivalente sin presión.

Variables: distancia en pies (0–36+), valor del tiro (2 o 3) y tipo de tiro (`sub_type`: Jump Shot, Layup, Dunk, etc.).

In [5]:
from src.analysis.run_analysis import fit_difficulty_model

base["p_esperada"], metrics, calibration = fit_difficulty_model(base)
display(pd.Series(metrics, name="valor").to_frame())
display(calibration)

,valor
auc_prueba,0.6580
log_loss_prueba,0.6424
log_loss_base,0.6915
tiros_entrenamiento,535554.0000
tiros_prueba,133889.0000
auc_clutch,0.6669


,decil,prob_media,acierto_real,tiros
0,0,0.3227,0.3099,13953
1,1,0.3589,0.3636,19467
2,2,0.3698,0.3687,9842
3,3,0.3838,0.3797,10732
4,4,0.3937,0.3996,13233
5,5,0.4244,0.4334,13695
6,6,0.4705,0.4740,13083
7,7,0.5297,0.5266,13123
8,8,0.6805,0.6703,16744
9,9,0.8371,0.8463,10017


## 5. Efecto clutch ajustado a nivel liga

In [6]:
import numpy as np

c = base[base["clutch"]]
obs, esp = c["is_made"].mean(), c["p_esperada"].mean()
se = np.sqrt((c["p_esperada"] * (1 - c["p_esperada"])).sum()) / len(c)
print(f"Efectividad real en clutch:     {obs:.2%}")
print(f"Efectividad esperada en clutch: {esp:.2%}")
print(f"Diferencia ajustada: {100*(obs-esp):.2f} pp (IC 95%: {100*(obs-esp-1.96*se):.2f} a {100*(obs-esp+1.96*se):.2f})")
print(f"Parte de la caída bruta explicada por dificultad: {(g.loc['No clutch','efectividad']-esp)/(g.loc['No clutch','efectividad']-obs):.0%}")

Efectividad real en clutch:     43.55%
Efectividad esperada en clutch: 46.17%
Diferencia ajustada: -2.62 pp (IC 95%: -3.23 a -2.01)
Parte de la caída bruta explicada por dificultad: 27%


## 6. Jugadores (preguntas secundarias 3, 4 y 5)

Para cada jugador con al menos 50 tiros clutch:
- **Efecto clutch ajustado** = (real − esperado) en clutch − (real − esperado) fuera de clutch. Compara al jugador contra *su propio nivel*, no contra la liga.
- **p-valor** con aproximación normal y **q-valor de Benjamini-Hochberg** para corregir por evaluar 162 jugadores a la vez.

In [7]:
from src.analysis.run_analysis import player_table

players = player_table(base)
print(f"Jugadores evaluados: {len(players)}")
print(f"p < 0.05 sin corregir: {(players['p_valor'] < 0.05).sum()} (por azar se esperarían ~{0.05*len(players):.0f})")
print(f"  de ellos positivos: {((players['p_valor'] < 0.05) & (players['efecto_clutch_ajustado'] > 0)).sum()}")
print(f"Significativos tras Benjamini-Hochberg: {players['significativo_bh'].sum()}")

cols = ["jugador", "equipo", "tiros_clutch", "efectividad_clutch", "esperada_clutch",
        "efecto_clutch_ajustado", "ic95_inf", "ic95_sup", "p_valor", "q_valor_bh"]
print("\nMayor efecto clutch ajustado:")
display(players[cols].head(10))
print("Menor efecto clutch ajustado:")
display(players[cols].tail(10))

Jugadores evaluados: 162
p < 0.05 sin corregir: 16 (por azar se esperarían ~8)
  de ellos positivos: 2
Significativos tras Benjamini-Hochberg: 0

Mayor efecto clutch ajustado:


,jugador,equipo,tiros_clutch,efectividad_clutch,esperada_clutch,efecto_clutch_ajustado,ic95_inf,ic95_sup,p_valor,q_valor_bh
0,Edgecombe,PHI,51,0.5686,0.4527,0.1497,0.0164,0.2830,0.0278,0.3834
1,Rollins,MIL,55,0.5636,0.4463,0.1026,-0.0271,0.2323,0.1210,0.5647
2,Grant,POR,100,0.5100,0.4520,0.0920,-0.0045,0.1885,0.0617,0.4349
3,Murray,DEN,204,0.5294,0.4326,0.0724,0.0052,0.1397,0.0348,0.3834
4,Gordon,DEN,86,0.5814,0.5210,0.0677,-0.0310,0.1664,0.1789,0.6704
5,Hield,ATL,69,0.4783,0.4231,0.0515,-0.0635,0.1664,0.3802,0.8603
6,O'Neale,PHX,61,0.4590,0.3872,0.0514,-0.0707,0.1736,0.4094,0.8603
7,Marshall,DAL,62,0.5484,0.4766,0.0443,-0.0756,0.1642,0.4692,0.8797
8,Strus,CLE,66,0.4242,0.3991,0.0425,-0.0748,0.1597,0.4777,0.8797
9,Monk,SAC,127,0.4882,0.4398,0.0419,-0.0433,0.1270,0.3349,0.8603


Menor efecto clutch ajustado:


,jugador,equipo,tiros_clutch,efectividad_clutch,esperada_clutch,efecto_clutch_ajustado,ic95_inf,ic95_sup,p_valor,q_valor_bh
152,Williamson,NOP,105,0.4667,0.5443,-0.1064,-0.1987,-0.0142,0.0237,0.3834
153,Butler III,GSW,121,0.3802,0.4781,-0.1073,-0.1956,-0.0190,0.0172,0.3834
154,Leonard,LAC,133,0.4211,0.4707,-0.1090,-0.1913,-0.0266,0.0095,0.3834
155,Embiid,PHI,103,0.3689,0.4626,-0.1162,-0.2119,-0.0206,0.0172,0.3834
156,Duren,DET,65,0.5846,0.6645,-0.1182,-0.2274,-0.0089,0.0340,0.3834
157,Jaquez Jr.,MIA,64,0.3438,0.4628,-0.1186,-0.2382,0.0010,0.0519,0.4349
158,Kuminga,ATL,53,0.3962,0.5250,-0.1191,-0.2480,0.0097,0.0699,0.4721
159,Jackson Jr.,MEM,147,0.3673,0.4793,-0.1239,-0.2036,-0.0442,0.0023,0.3736
160,LaVine,SAC,93,0.3548,0.4414,-0.1283,-0.2273,-0.0293,0.0110,0.3834
161,Dort,OKC,54,0.2407,0.4289,-0.1728,-0.3029,-0.0427,0.0092,0.3834


### Superar a la liga ≠ ser clutch

Algunos jugadores encestan en clutch más de lo que esperaría un tiro promedio de la liga (`z_vs_liga`), pero también lo hacen fuera de clutch: es su nivel general, no un efecto de presión.

In [8]:
display(players.nlargest(5, "z_vs_liga")[["jugador", "tiros_clutch", "efectividad_clutch",
        "esperada_clutch", "z_vs_liga", "ventaja_ajustada_no_clutch", "efecto_clutch_ajustado", "q_valor_bh"]])

,jugador,tiros_clutch,efectividad_clutch,esperada_clutch,z_vs_liga,ventaja_ajustada_no_clutch,efecto_clutch_ajustado,q_valor_bh
3,Murray,204,0.5294,0.4326,2.8963,0.0244,0.0724,0.3834
10,Gilgeous-Alexander,248,0.5323,0.4485,2.7410,0.0431,0.0406,0.6704
1,Rollins,55,0.5636,0.4463,1.8107,0.0147,0.1026,0.5647
0,Edgecombe,51,0.5686,0.4527,1.7422,-0.0338,0.1497,0.3834
88,Jokić,273,0.5201,0.4738,1.5959,0.0823,-0.0360,0.7312


## 7. Observaciones

- La caída de efectividad en clutch es real a nivel liga y solo una parte se explica por tiros más difíciles.
- Con 3 temporadas, ningún jugador muestra un efecto clutch individual que sobreviva a la corrección por comparaciones múltiples.
- Las conclusiones finales y las figuras están en `04_results.ipynb` y en `docs/final_report.md`.